# Phone Studio — ComfyUI on Google Colab

This notebook is the GPU half of the Phone Studio Android app. It keeps models and results in Google Drive while running ComfyUI itself from Colab's fast local disk.

## Before running

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU available to your account). Colab does not have a supported API for an app to reserve a GPU automatically.
2. Add `NGROK_AUTHTOKEN` in **Colab Secrets**. An ngrok account/token is required for the phone to reach the temporary Colab machine.
   Optionally also add `NGROK_DOMAIN` (a free dev domain from ngrok dashboard → Gateway → Domains) so the Server URL stays the same every time this notebook runs.
3. Optionally add `CIVITAI_API_KEY` in Colab Secrets. It remains in Colab and lets the app ask the server to download Civitai model versions directly to Drive.
4. Run every cell. The final cell first warms up your existing **WAI NSFW Illustrious v170** checkpoint with a 1024 x 1024, two-step landscape generation. Wait for **Warm-up complete**, then copy the four connection values printed by that cell into the Phone Studio app's **Connect** screen.

Keep this notebook running while using the phone app. Do not share the ngrok URL, tunnel password, or Phone Studio token.

## Faster T4 generation

- Selected checkpoints, LoRAs and other weights loaded through ComfyUI's standard loader are copied to `/content/phone_studio_model_cache` on first use. Later loads read the local copy. Drive remains the permanent model store; downloads from the app still go to Drive. The first copy takes time, and Colab resets clear the cache.
- The cache is capped at **24 GiB**, leaves **8 GiB** free, and falls back to Drive if it cannot cache a model. It never deletes Drive models or evicts files that a running model might still use. Adjust the limits in the GPU settings cell.
- FP16 diffusion/text encoding and cuDNN autotuning are enabled. `CHANNELS_LAST` is enabled for WAI Illustrious/SDXL; turn it off if switching to an unfamiliar architecture. Image dimensions, sampling steps, prompts and LoRA strengths are unchanged.
- Automatic `torch.compile` is off by default and skipped on T4: current Triton requires compute capability 8.0+, while T4 is 7.5. The `PHONE_STUDIO_AUTO_COMPILE=1` secret opts in on supported newer GPUs; compilation can still make the first image much slower.
- `VRAM_MODE = 'auto'` lets ComfyUI retain weights when possible and free memory when needed. Try `'high'` only if your complete workflow fits in VRAM; it is not automatically faster.

To compare speed, use the same checkpoint, LoRAs, dimensions, sampler and step count. Record the first image separately (model copy/load and autotuning), then compare the **Prompt executed in ... seconds** logs for three more images with different seeds so ComfyUI actually resamples. No T4 timing has been measured for these changes yet.

Implementation references: [ComfyUI options](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/cli_args.py), [ComfyUI model loading](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/utils.py), [sample-time compilation helper](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy_api/torch_helpers/torch_compile.py), [Triton GPU support](https://github.com/triton-lang/triton#compatibility).

## Default checkpoint and startup warm-up

The default is your existing **`waiIllustriousSDXL_v170.safetensors`** under Drive's `PhoneStudio/models/checkpoints`. No checkpoint is downloaded. If you move it into a subfolder, include that prefix in `DEFAULT_CHECKPOINT`. Clearing this setting enables automatic WAI v170 name matching.

The bridge lists this checkpoint first, exposes `defaultCheckpoint`, and fills missing checkpoint choices. A checkpoint explicitly selected or remembered by the phone app still takes precedence. The startup generation always uses WAI v170 with no LoRAs, and completes checkpoint loading, both text encodings, sampling and VAE decoding before printing the connection details. Its temporary preview stays on Colab's local disk and does not replace the phone's latest job.

Warm-up shifts the initial loading cost to notebook startup. ComfyUI retains cached weights subject to available RAM/VRAM; changing models, adding LoRAs, larger images or memory pressure can still trigger loading. Set the warm-up dimensions to your usual image size for more relevant cuDNN tuning.


In [ ]:
# Mount Drive and establish persistent folders and private credentials.
import json
import os
import secrets
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

def colab_secret(name: str) -> str:
    value = os.environ.get(name, '').strip()
    if value:
        return value
    try:
        return (userdata.get(name) or '').strip()
    except Exception:
        return ''

DRIVE_ROOT = Path('/content/drive/MyDrive/PhoneStudio')
MODELS_ROOT = DRIVE_ROOT / 'models'
OUTPUT_ROOT = DRIVE_ROOT / 'output'
INPUT_ROOT = DRIVE_ROOT / 'input'
USER_ROOT = DRIVE_ROOT / 'user'
CUSTOM_NODES_ROOT = DRIVE_ROOT / 'custom_nodes'
RUNTIME_ROOT = Path('/content/ComfyUI')
for folder in (MODELS_ROOT, OUTPUT_ROOT, INPUT_ROOT, USER_ROOT, CUSTOM_NODES_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
for folder in ('checkpoints', 'loras', 'vae', 'upscale_models', 'controlnet'):
    (MODELS_ROOT / folder).mkdir(parents=True, exist_ok=True)

credentials_path = DRIVE_ROOT / '.phone_studio_credentials.json'
try:
    stored_credentials = json.loads(credentials_path.read_text())
except (FileNotFoundError, json.JSONDecodeError):
    stored_credentials = {}

PHONE_STUDIO_TOKEN = colab_secret('PHONE_STUDIO_TOKEN') or stored_credentials.get('phone_studio_token') or secrets.token_urlsafe(32)
TUNNEL_PASSWORD = colab_secret('TUNNEL_PASSWORD') or stored_credentials.get('tunnel_password') or secrets.token_urlsafe(24)
TUNNEL_USER = 'studio'
credentials_path.write_text(json.dumps({'phone_studio_token': PHONE_STUDIO_TOKEN, 'tunnel_password': TUNNEL_PASSWORD}, indent=2))

NGROK_AUTHTOKEN = colab_secret('NGROK_AUTHTOKEN')
CIVITAI_API_KEY = colab_secret('CIVITAI_API_KEY')
# Opt in only for repeated runs on a supported GPU; the T4 path skips compilation.
PHONE_STUDIO_AUTO_COMPILE = colab_secret('PHONE_STUDIO_AUTO_COMPILE') or '0'
if not NGROK_AUTHTOKEN:
    raise RuntimeError('Add NGROK_AUTHTOKEN in Colab Secrets, then rerun this cell.')

print(f'Drive workspace: {DRIVE_ROOT}')
print('Persistent models:', MODELS_ROOT)


In [ ]:
# Install ComfyUI on the fast runtime disk and link large/persistent folders to Drive.
import shutil
import subprocess
import sys

def run(command, **kwargs):
    print('$', ' '.join(map(str, command)))
    return subprocess.run(command, check=True, **kwargs)

def link_drive_folder(link: Path, target: Path):
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.parent.mkdir(parents=True, exist_ok=True)
    link.symlink_to(target, target_is_directory=True)

run(['apt-get', 'update', '-y'])
run(['apt-get', 'install', '-y', 'git', 'libgl1', 'libglib2.0-0'])
if not RUNTIME_ROOT.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(RUNTIME_ROOT)])
else:
    run(['git', '-C', str(RUNTIME_ROOT), 'pull', '--ff-only'])

link_drive_folder(RUNTIME_ROOT / 'models', MODELS_ROOT)
link_drive_folder(RUNTIME_ROOT / 'output', OUTPUT_ROOT)
link_drive_folder(RUNTIME_ROOT / 'input', INPUT_ROOT)
link_drive_folder(RUNTIME_ROOT / 'user', USER_ROOT)
link_drive_folder(RUNTIME_ROOT / 'custom_nodes', CUSTOM_NODES_ROOT)

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(RUNTIME_ROOT / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok', 'requests'])
print('ComfyUI base is ready.')


In [ ]:
# GPU settings for Colab. No replacement of Colab's CUDA/PyTorch installation.
import torch

LOCAL_MODEL_CACHE = True
MODEL_CACHE_MAX_GB = 24
MODEL_CACHE_RESERVE_GB = 8
MODEL_CACHE_ROOT = Path('/content/phone_studio_model_cache')
VRAM_MODE = 'auto'       # 'high' is optional for workflows known to fit fully in VRAM.
CHANNELS_LAST = True    # WAI Illustrious is an SDXL image checkpoint.
                       # Set False if switching to unfamiliar/video architectures with 5D weights.

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun the notebook.')
if VRAM_MODE not in {'auto', 'high'}:
    raise ValueError("VRAM_MODE must be 'auto' or 'high'.")
if MODEL_CACHE_MAX_GB <= 0 or MODEL_CACHE_RESERVE_GB < 0:
    raise ValueError('Cache capacity must be positive and reserved disk space cannot be negative.')

SAGE_ATTENTION_AVAILABLE = False
cc_major, cc_minor = torch.cuda.get_device_capability(0)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} | {vram_gb:.1f} GiB VRAM | compute capability {cc_major}.{cc_minor}')

if cc_major >= 8:
    print('Attempting SageAttention2 on supported hardware (installation can take a few minutes)...')
    try:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'sageattention==2.2.0', '--no-build-isolation'])
        import sageattention
        SAGE_ATTENTION_AVAILABLE = True
        print('SageAttention2 installed; using --use-sage-attention.')
    except Exception as exc:
        print(f'SageAttention2 unavailable ({exc}); using PyTorch attention.')
else:
    print('T4/pre-Ampere: using PyTorch attention; skipping unsupported SageAttention2 kernels.')
    if PHONE_STUDIO_AUTO_COMPILE == '1':
        print('Skipping automatic compilation: current Triton requires compute capability 8.0+.')
    PHONE_STUDIO_AUTO_COMPILE = '0'

print(f'Local model cache: {"on" if LOCAL_MODEL_CACHE else "off"}; maximum {MODEL_CACHE_MAX_GB} GiB.')
print(f'VRAM mode: {VRAM_MODE}; automatic compilation: {PHONE_STUDIO_AUTO_COMPILE == "1"}.')

# Default model already stored in Drive: no download is needed.
# Exact existing checkpoint, relative to Drive's models/checkpoints folder.
# Include the subfolder prefix here if you move it into a subfolder.
DEFAULT_CHECKPOINT = 'waiIllustriousSDXL_v170.safetensors'
WARMUP_ON_START = True
WARMUP_WIDTH = 1024
WARMUP_HEIGHT = 1024
WARMUP_STEPS = 2
WARMUP_TIMEOUT_SECONDS = 900
WARMUP_PROMPT = 'a quiet mountain lake at sunrise, landscape, anime illustration'


def resolve_default_checkpoint(checkpoint_root, requested=''):
    import re
    available = sorted(
        item.relative_to(checkpoint_root).as_posix()
        for item in checkpoint_root.rglob('*')
        if item.is_file() and item.suffix.lower() in {'.safetensors', '.ckpt', '.pt'}
    )
    if requested:
        if requested not in available:
            raise FileNotFoundError(f'Default checkpoint is not in {checkpoint_root}: {requested}')
        return requested
    matches = []
    for name in available:
        normalized = re.sub(r'[^a-z0-9]', '', Path(name).stem.lower())
        if ('wai' in normalized and 'illustrious' in normalized
                and re.search(r'v170(?![0-9])', normalized)):
            matches.append(name)
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f'Could not identify WAI Illustrious v170 in {checkpoint_root}. '
            'Set DEFAULT_CHECKPOINT to its exact filename above. '
            f'Available checkpoints: {available}'
        )
    raise ValueError(
        'Several WAI Illustrious v170 files were found. '
        f'Set DEFAULT_CHECKPOINT to one of these exact names: {matches}'
    )

DEFAULT_CHECKPOINT = resolve_default_checkpoint(MODELS_ROOT / 'checkpoints', DEFAULT_CHECKPOINT)
print('Default checkpoint:', DEFAULT_CHECKPOINT)
print(f'Startup warm-up: {WARMUP_ON_START} ({WARMUP_WIDTH} x {WARMUP_HEIGHT}, {WARMUP_STEPS} steps).')


In [ ]:
# Optional: NVIDIA TensorRT can be very fast for a fixed, separately converted
# checkpoint, but it is not used by Phone Studio's normal CheckpointLoaderSimple
# workflow and adds a slow install to every fresh Colab session. Leave this off
# unless you will build and use a TensorRT workflow from the ComfyUI web UI.
INSTALL_TENSORRT = False
TENSORRT_NODE_DIR = CUSTOM_NODES_ROOT / 'ComfyUI_TensorRT'

if INSTALL_TENSORRT:
    try:
        if not TENSORRT_NODE_DIR.exists():
            run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI_TensorRT.git', str(TENSORRT_NODE_DIR)])
        run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(TENSORRT_NODE_DIR / 'requirements.txt')])
        print('ComfyUI_TensorRT node installed. Build a TensorRT workflow in the ComfyUI web UI to use it.')
    except Exception as exc:
        print(f'TensorRT node install failed ({exc}) — skipping it; normal Phone Studio generation still works.')
else:
    print('TensorRT install skipped. Set INSTALL_TENSORRT = True only for a separately converted TensorRT workflow.')

In [ ]:
# Add Phone Studio's small authenticated API as a ComfyUI custom node.
# It lists local models and downloads a Civitai *model version* directly to Drive.
BRIDGE_DIR = CUSTOM_NODES_ROOT / 'phone_studio_bridge'
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)

BRIDGE_CODE = r'''
import asyncio
import hmac
import os
import re
import uuid
from pathlib import Path

import requests
import torch
from aiohttp import web
from server import PromptServer

MODELS_ROOT = Path(os.environ.get('PHONE_STUDIO_MODELS_ROOT', '/content/drive/MyDrive/PhoneStudio/models'))
PHONE_TOKEN = os.environ.get('PHONE_STUDIO_TOKEN', '')
CIVITAI_KEY = os.environ.get('CIVITAI_API_KEY', '')
DEFAULT_CHECKPOINT = os.environ.get('PHONE_STUDIO_DEFAULT_CHECKPOINT', '')
DOWNLOADS = {}
INSTANCE_ID = uuid.uuid4().hex
routes = PromptServer.instance.routes

# --- Read model weights from Colab's local disk after one sequential copy ---
# Keep model discovery and Civitai downloads on Drive. Intercept only the
# standard weight loader, so checkpoint/LoRA names and workflow APIs stay intact.
import functools
import hashlib
import shutil
import threading
import time
import comfy.utils

CACHE_ROOT = Path(os.environ.get('PHONE_STUDIO_MODEL_CACHE_ROOT', '/content/phone_studio_model_cache'))
CACHE_ENABLED = os.environ.get('PHONE_STUDIO_LOCAL_MODEL_CACHE', '1') == '1'
CACHE_MAX_BYTES = int(float(os.environ.get('PHONE_STUDIO_MODEL_CACHE_MAX_GB', '24')) * 1024**3)
CACHE_RESERVE_BYTES = int(float(os.environ.get('PHONE_STUDIO_MODEL_CACHE_RESERVE_GB', '8')) * 1024**3)
_CACHE_LOCK = threading.Lock()


def local_model_path(filename):
    if not CACHE_ENABLED:
        return filename
    temp = None
    try:
        source = Path(filename).resolve()
        # Do not cache arbitrary uploads, outputs, or files outside the model store.
        source.relative_to(MODELS_ROOT.resolve())
        if source.suffix.lower() not in {'.safetensors', '.sft', '.ckpt', '.pt', '.pth', '.bin'}:
            return filename
        with _CACHE_LOCK:
            stat = source.stat()
            # A replacement model gets a different local file, even if its name
            # stays the same. Never overwrite a file that may be memory-mapped.
            signature = f'{source}|{stat.st_size}|{stat.st_mtime_ns}'
            key = hashlib.sha256(signature.encode()).hexdigest()
            destination = CACHE_ROOT / key / source.name
            if destination.is_file() and destination.stat().st_size == stat.st_size:
                return str(destination)
            CACHE_ROOT.mkdir(parents=True, exist_ok=True)
            used = sum(item.stat().st_size for item in CACHE_ROOT.rglob('*') if item.is_file())
            free = shutil.disk_usage(CACHE_ROOT).free
            if used + stat.st_size > CACHE_MAX_BYTES or free - stat.st_size < CACHE_RESERVE_BYTES:
                print(f'[phone-studio] Local cache full/low disk; loading {source.name} from Drive.')
                return filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            temp = destination.with_name(destination.name + f'.{uuid.uuid4().hex}.part')
            started = time.perf_counter()
            print(f'[phone-studio] Caching {source.name} ({stat.st_size / 1024**3:.2f} GiB) on local disk...')
            shutil.copyfile(source, temp)
            after = source.stat()
            if (after.st_size, after.st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):
                raise OSError('Source model changed during copying; retry on the next load.')
            if temp.stat().st_size != stat.st_size:
                raise OSError('Incomplete model cache copy.')
            temp.replace(destination)
            print(f'[phone-studio] Local copy ready in {time.perf_counter() - started:.1f}s; reused on later loads.')
            return str(destination)
    except ValueError:
        return filename  # Model is outside the persistent models directory.
    except OSError as exc:
        print(f'[phone-studio] Local cache unavailable ({exc}); using the original model file.')
        return filename
    finally:
        if temp is not None:
            try:
                temp.unlink(missing_ok=True)
            except OSError:
                pass


_drive_load_torch_file = comfy.utils.load_torch_file

@functools.wraps(_drive_load_torch_file)
def _load_torch_file_cached(ckpt, *args, **kwargs):
    return _drive_load_torch_file(local_model_path(ckpt), *args, **kwargs)

comfy.utils.load_torch_file = _load_torch_file_cached
print(f'[phone-studio] Local model cache {"enabled" if CACHE_ENABLED else "disabled"}.')

# --- Optional compilation for newer supported GPUs, disabled on T4 ---------
# Use ComfyUI's sample-time wrapper to preserve normal parameter names for
# LoRA loading/patching. Never permanently replace diffusion_model itself.
if os.environ.get('PHONE_STUDIO_AUTO_COMPILE', '0') == '1' and torch.cuda.is_available():
    if torch.cuda.get_device_capability(0)[0] < 8:
        print('[phone-studio] Skipping torch.compile on T4/pre-Ampere (current Triton support starts at 8.0).')
    else:
        try:
            import torch._dynamo
            import comfy.sd
            from comfy_api.torch_helpers import set_torch_compile_wrapper

            torch._dynamo.config.suppress_errors = True
            _original_load_checkpoint = comfy.sd.load_checkpoint_guess_config

            @functools.wraps(_original_load_checkpoint)
            def _load_checkpoint_and_compile(*args, **kwargs):
                result = _original_load_checkpoint(*args, **kwargs)
                if not result or result[0] is None:
                    return result
                try:
                    compiled_patcher = result[0].clone(disable_dynamic=True)
                    set_torch_compile_wrapper(
                        model=compiled_patcher, backend='inductor', mode='default',
                        fullgraph=False, dynamic=None,
                    )
                    print('[phone-studio] Optional sample-time compilation enabled; first sample may be slow.')
                    return (compiled_patcher, *result[1:])
                except Exception as exc:
                    print(f'[phone-studio] Compilation setup skipped: {exc}')
                    return result

            comfy.sd.load_checkpoint_guess_config = _load_checkpoint_and_compile
        except Exception as exc:
            print(f'[phone-studio] Compilation helper unavailable ({exc}); using normal execution.')

# Fill only missing checkpoint selections. Explicit phone/UI choices win.
def _phone_studio_default_checkpoint(json_data):
    prompt = json_data.get('prompt')
    if DEFAULT_CHECKPOINT and isinstance(prompt, dict):
        for node in prompt.values():
            if not isinstance(node, dict) or node.get('class_type') != 'CheckpointLoaderSimple':
                continue
            inputs = node.get('inputs')
            if inputs is None:
                inputs = node['inputs'] = {}
            if not isinstance(inputs, dict):
                continue
            selected = inputs.get('ckpt_name')
            if selected is None or (isinstance(selected, str) and not selected.strip()):
                inputs['ckpt_name'] = DEFAULT_CHECKPOINT
    return json_data

PromptServer.instance.add_on_prompt_handler(_phone_studio_default_checkpoint)

def require_token(request):
    supplied = request.headers.get('X-Phone-Studio-Token', '')
    if not PHONE_TOKEN or not hmac.compare_digest(supplied, PHONE_TOKEN):
        raise web.HTTPUnauthorized(text='Phone Studio token required')

def safe_filename(value):
    value = Path(value).name
    value = re.sub(r'[^A-Za-z0-9._ -]+', '_', value).strip(' .')
    if not value.lower().endswith('.safetensors'):
        value += '.safetensors'
    if not value or value == '.safetensors':
        raise ValueError('Model file name is invalid')
    return value

def models_in(folder):
    path = MODELS_ROOT / folder
    if not path.exists():
        return []
    return sorted(str(item.relative_to(path)).replace('\\', '/') for item in path.rglob('*') if item.is_file() and item.suffix.lower() in {'.safetensors', '.ckpt', '.pt'})

@routes.get('/phone-studio/health')
async def phone_studio_health(request):
    require_token(request)
    return web.json_response({'ok': True, 'service': 'phone-studio-bridge', 'instanceId': INSTANCE_ID})

@routes.get('/phone-studio/models')
async def phone_studio_models(request):
    require_token(request)
    checkpoints = models_in('checkpoints')
    if DEFAULT_CHECKPOINT in checkpoints:
        checkpoints.remove(DEFAULT_CHECKPOINT)
        checkpoints.insert(0, DEFAULT_CHECKPOINT)
    return web.json_response({
        'checkpoints': checkpoints, 'loras': models_in('loras'),
        'defaultCheckpoint': DEFAULT_CHECKPOINT,
    })

def download_civitai(job_id, version_id, requested_kind):
    state = DOWNLOADS[job_id]
    headers = {'Authorization': f'Bearer {CIVITAI_KEY}'} if CIVITAI_KEY else {}
    try:
        state.update(status='looking up model', progress=0)
        response = requests.get(f'https://civitai.com/api/v1/model-versions/{version_id}', headers=headers, timeout=45)
        response.raise_for_status()
        info = response.json()
        files = info.get('files') or []
        file_info = next((item for item in files if item.get('primary')), None) or next((item for item in files if str(item.get('name', '')).lower().endswith('.safetensors')), None)
        if not file_info or not file_info.get('downloadUrl'):
            raise RuntimeError('Civitai did not return a downloadable SafeTensors file for this version')
        inferred_type = str((info.get('model') or {}).get('type', '')).lower()
        kind = requested_kind if requested_kind in {'checkpoint', 'lora'} else ('lora' if 'lora' in inferred_type else 'checkpoint')
        destination_dir = MODELS_ROOT / ('loras' if kind == 'lora' else 'checkpoints')
        destination_dir.mkdir(parents=True, exist_ok=True)
        filename = safe_filename(file_info.get('name') or f'civitai_{version_id}.safetensors')
        destination = destination_dir / filename
        temp = destination.with_suffix(destination.suffix + '.part')
        state.update(status='downloading', filename=filename, progress=0)
        with requests.get(file_info['downloadUrl'], headers=headers, stream=True, allow_redirects=True, timeout=90) as stream:
            stream.raise_for_status()
            total = int(stream.headers.get('content-length') or 0)
            downloaded = 0
            with open(temp, 'wb') as handle:
                for chunk in stream.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    handle.write(chunk)
                    downloaded += len(chunk)
                    state['progress'] = round(downloaded * 100 / total, 1) if total else None
        temp.replace(destination)
        state.update(status='complete', progress=100, filename=filename, kind=kind)
    except Exception as exc:
        state.update(status='failed', error=str(exc))

@routes.post('/phone-studio/download-model')
async def phone_studio_download(request):
    require_token(request)
    payload = await request.json()
    version_id = str(payload.get('modelVersionId', '')).strip()
    if not version_id.isdigit():
        raise web.HTTPBadRequest(text='modelVersionId must be a Civitai model-version ID')
    requested_kind = str(payload.get('kind', '')).lower()
    job_id = uuid.uuid4().hex
    DOWNLOADS[job_id] = {'status': 'queued', 'progress': 0}
    asyncio.create_task(asyncio.to_thread(download_civitai, job_id, version_id, requested_kind))
    return web.json_response({'jobId': job_id})

@routes.get('/phone-studio/downloads/{job_id}')
async def phone_studio_download_status(request):
    require_token(request)
    state = DOWNLOADS.get(request.match_info['job_id'])
    if not state:
        raise web.HTTPNotFound(text='Download job not found')
    return web.json_response(state)

# --- Track + expose the most recent generation job -------------------------
# Nothing here ever cancels a running job: ComfyUI already executes queued
# prompts to completion independent of whether any client is connected. The
# phone app talks to ComfyUI's own /prompt and /ws directly for generation
# (this bridge never sees those calls), so if a job looks "cancelled" after
# backgrounding the app, it's almost certainly the app's live connection that
# died, not the job. This just gives the app a stable, tokenized way to ask
# "what happened to my last job" after regaining network, instead of only
# ever finding out live over the /ws socket it may have lost.
import time as _time

LATEST_JOB = {'prompt_id': None, 'submitted_at': None, 'client_id': None, 'request_id': None}

@web.middleware
async def _phone_studio_track_prompt(request, handler):
    body_in = {}
    if request.method == 'POST' and request.path == '/prompt':
        try:
            body_in = await request.json()
        except Exception:
            body_in = {}
    response = await handler(request)
    is_warmup = bool((body_in.get('extra_data') or {}).get('phone_studio_warmup'))
    if request.method == 'POST' and request.path == '/prompt' and response.status == 200 and not is_warmup:
        try:
            import json as _json
            prompt_id = _json.loads(response.text).get('prompt_id')
            if prompt_id:
                LATEST_JOB['prompt_id'] = prompt_id
                LATEST_JOB['submitted_at'] = _time.time()
                LATEST_JOB['client_id'] = str(body_in.get('client_id') or '')
                LATEST_JOB['request_id'] = str((body_in.get('extra_data') or {}).get('phone_studio_request_id') or '')
        except Exception:
            pass
    return response
try:
    PromptServer.instance.app.middlewares.append(_phone_studio_track_prompt)
    print('[phone-studio] latest-job tracking enabled')
except Exception as exc:
    print(f'[phone-studio] could not install latest-job tracking ({exc}); /phone-studio/latest-job will always report none')

@routes.get('/phone-studio/latest-job')
async def phone_studio_latest_job(request):
    require_token(request)
    requested_client_id = request.query.get('clientId', '')
    requested_request_id = request.query.get('requestId', '')
    if requested_client_id and requested_client_id != LATEST_JOB.get('client_id'):
        return web.json_response({'status': 'none'})
    if requested_request_id and requested_request_id != LATEST_JOB.get('request_id'):
        return web.json_response({'status': 'none'})
    prompt_id = LATEST_JOB.get('prompt_id')
    if not prompt_id:
        return web.json_response({'status': 'none'})
    submitted_at = LATEST_JOB.get('submitted_at')
    request_id = LATEST_JOB.get('request_id')
    try:
        history = PromptServer.instance.prompt_queue.get_history(prompt_id=prompt_id)
    except Exception as exc:
        return web.json_response({'status': 'unknown', 'promptId': prompt_id, 'requestId': request_id, 'submittedAt': submitted_at, 'error': str(exc)})
    entry = history.get(prompt_id)
    if entry is None:
        return web.json_response({'status': 'pending', 'promptId': prompt_id, 'requestId': request_id, 'submittedAt': submitted_at})
    images = []
    for node_output in (entry.get('outputs') or {}).values():
        images.extend(node_output.get('images') or [])
    status_info = entry.get('status') or {}
    failed = str(status_info.get('status_str', '')).lower() == 'error'
    return web.json_response({'status': 'error' if failed else 'done', 'promptId': prompt_id, 'requestId': request_id, 'submittedAt': submitted_at, 'images': images})
# --- Replay a missed result over /ws when the phone reconnects -------------
# Best-effort, and it rests on one assumption about the app: that it reuses a
# stable clientId across reconnects (the standard ComfyUI client pattern -
# generate a UUID once, keep sending it as ?clientId=... on /ws and client_id
# in the /prompt body) rather than minting a new one every time it opens a
# socket. If it does, this makes a finished-while-offline job show up the
# instant the socket reopens, without the app doing anything different.
# If it doesn't reuse a clientId, this silently does nothing extra - it
# can't identify "the same phone" on reconnect, so there's nothing to key
# the replay on. Check the Colab log line below to see what your app is
# actually doing here; that tells you in one glance whether this can work
# at all before you rely on it.
#
# What it actually sends over the socket is a synthetic version of ComfyUI's
# own live 'executed' / 'executing' messages, reconstructed from history
# after the fact. I can't test this against your specific app from here -
# confirm it works by starting a generation, backgrounding the app before
# it finishes, waiting past completion, then reopening the app with no other
# action and checking whether the image just appears.
PENDING_REPLAY = {}

async def _phone_studio_watch_job(client_id, prompt_id):
    if not client_id or not prompt_id:
        return
    for _ in range(600):  # ~10 min ceiling; real generations finish well inside this
        try:
            history = PromptServer.instance.prompt_queue.get_history(prompt_id=prompt_id)
        except Exception:
            return
        entry = history.get(prompt_id)
        if entry is not None:
            if PromptServer.instance.sockets.get(client_id) is None:
                PENDING_REPLAY[client_id] = {'prompt_id': prompt_id, 'outputs': entry.get('outputs') or {}}
                print(f'[phone-studio] {client_id} was offline when {prompt_id} finished; queued for replay on reconnect')
            return
        await asyncio.sleep(1)

async def _phone_studio_replay_when_ready(client_id):
    for _ in range(50):  # ~5s to let the handshake finish and register the socket
        ws = PromptServer.instance.sockets.get(client_id)
        if ws is not None:
            break
        await asyncio.sleep(0.1)
    else:
        return
    payload = PENDING_REPLAY.pop(client_id, None)
    if not payload:
        return
    try:
        for node_id, node_output in (payload['outputs'] or {}).items():
            await ws.send_json({'type': 'executed', 'data': {'node': node_id, 'output': node_output, 'prompt_id': payload['prompt_id']}})
        await ws.send_json({'type': 'executing', 'data': {'node': None, 'prompt_id': payload['prompt_id']}})
        print(f"[phone-studio] replayed missed result to {client_id}")
    except Exception as exc:
        print(f'[phone-studio] replay to {client_id} failed: {exc}')

@web.middleware
async def _phone_studio_reconnect_bridge(request, handler):
    if request.method == 'POST' and request.path == '/prompt':
        try:
            body_in = await request.json()
            client_id = body_in.get('client_id')
            is_warmup = bool((body_in.get('extra_data') or {}).get('phone_studio_warmup'))
        except Exception:
            client_id = None
            is_warmup = False
        response = await handler(request)
        if client_id and not is_warmup and response.status == 200:
            try:
                import json as _json
                prompt_id = _json.loads(response.text).get('prompt_id')
                if prompt_id:
                    asyncio.create_task(_phone_studio_watch_job(client_id, prompt_id))
            except Exception:
                pass
        return response
    if request.method == 'GET' and request.path == '/ws':
        client_id = request.query.get('clientId')
        print(f'[phone-studio] /ws connect, clientId={client_id!r}, pending replay={client_id in PENDING_REPLAY}')
        if client_id and client_id in PENDING_REPLAY:
            asyncio.create_task(_phone_studio_replay_when_ready(client_id))
    return await handler(request)
try:
    PromptServer.instance.app.middlewares.append(_phone_studio_reconnect_bridge)
    print('[phone-studio] reconnect-replay bridge enabled')
except Exception as exc:
    print(f'[phone-studio] could not install reconnect-replay bridge ({exc}); reconnects behave as before')

NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
'''
(BRIDGE_DIR / '__init__.py').write_text(BRIDGE_CODE)
print('Phone Studio bridge written to', BRIDGE_DIR)


In [ ]:
# Start ComfyUI and create a password-protected ngrok endpoint.
import socket
import time
import requests

from pyngrok import ngrok

def warmup_default_checkpoint(base_url):
    # PreviewImage is an output node: this runs VAE decoding too, while writing
    # only a disposable local preview, never a warm-up image to the Drive gallery.
    prompt = {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': DEFAULT_CHECKPOINT}},
        '2': {'class_type': 'CLIPTextEncode', 'inputs': {'text': WARMUP_PROMPT, 'clip': ['1', 1]}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['1', 1]}},
        '4': {'class_type': 'EmptyLatentImage', 'inputs': {
            'width': WARMUP_WIDTH, 'height': WARMUP_HEIGHT, 'batch_size': 1,
        }},
        '5': {'class_type': 'KSampler', 'inputs': {
            'model': ['1', 0], 'seed': 170, 'steps': WARMUP_STEPS, 'cfg': 5.0,
            'sampler_name': 'euler_ancestral', 'scheduler': 'normal',
            'positive': ['2', 0], 'negative': ['3', 0], 'latent_image': ['4', 0], 'denoise': 1.0,
        }},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5', 0], 'vae': ['1', 2]}},
        '7': {'class_type': 'PreviewImage', 'inputs': {'images': ['6', 0]}},
    }
    started = time.monotonic()
    print(f'Warming up {DEFAULT_CHECKPOINT}: {WARMUP_WIDTH} x {WARMUP_HEIGHT}, {WARMUP_STEPS} steps...')
    with requests.Session() as session:
        session.headers.update({'X-Phone-Studio-Token': PHONE_STUDIO_TOKEN})
        response = session.post(base_url + '/prompt', json={
            'prompt': prompt,
            'client_id': 'phone-studio-startup-warmup',
            'extra_data': {'phone_studio_warmup': True},
        }, timeout=30)
        if response.status_code != 200:
            raise RuntimeError(f'Warm-up prompt was rejected: {response.text[:1500]}')
        prompt_id = response.json().get('prompt_id')
        if not prompt_id:
            raise RuntimeError(f'Warm-up did not receive a prompt ID: {response.text[:1500]}')
        next_update = started + 15
        while time.monotonic() - started < WARMUP_TIMEOUT_SECONDS:
            if comfy_process.poll() is not None:
                raise RuntimeError('ComfyUI exited during model warm-up. Inspect the output above.')
            response = session.get(base_url + f'/history/{prompt_id}', timeout=30)
            response.raise_for_status()
            entry = response.json().get(prompt_id)
            if entry is not None:
                status = entry.get('status') or {}
                if status.get('status_str') == 'error':
                    raise RuntimeError(f'Model warm-up failed: {str(status.get("messages", []))[:2000]}')
                if status.get('completed'):
                    if not any(output.get('images') for output in (entry.get('outputs') or {}).values()):
                        raise RuntimeError('Warm-up completed without a decoded preview; inspect the ComfyUI log.')
                    print(f'Warm-up complete in {time.monotonic() - started:.1f}s. Default model is ready.')
                    return
                raise RuntimeError(f'Warm-up stopped before completion: {str(status)[:1500]}')
            if time.monotonic() >= next_update:
                print(f'Warm-up still running ({time.monotonic() - started:.0f}s); first-time model loading can take a while.')
                next_update = time.monotonic() + 15
            time.sleep(2)
    raise TimeoutError('Model warm-up timed out. Inspect the log or increase WARMUP_TIMEOUT_SECONDS.')


PORT = 8188
env = os.environ.copy()
env.update({
    'PHONE_STUDIO_MODELS_ROOT': str(MODELS_ROOT),
    'PHONE_STUDIO_DEFAULT_CHECKPOINT': DEFAULT_CHECKPOINT,
    'PHONE_STUDIO_TOKEN': PHONE_STUDIO_TOKEN,
    'CIVITAI_API_KEY': CIVITAI_API_KEY,
    'PHONE_STUDIO_AUTO_COMPILE': PHONE_STUDIO_AUTO_COMPILE,
    'PHONE_STUDIO_LOCAL_MODEL_CACHE': '1' if LOCAL_MODEL_CACHE else '0',
    'PHONE_STUDIO_MODEL_CACHE_ROOT': str(MODEL_CACHE_ROOT),
    'PHONE_STUDIO_MODEL_CACHE_MAX_GB': str(MODEL_CACHE_MAX_GB),
    'PHONE_STUDIO_MODEL_CACHE_RESERVE_GB': str(MODEL_CACHE_RESERVE_GB),
})

attention_flag = ['--use-sage-attention'] if SAGE_ATTENTION_AVAILABLE else ['--use-pytorch-cross-attention']
print('Attention backend:', attention_flag[0])

# Stop an earlier server from this notebook before allocating another model.
previous_process = globals().get('comfy_process')
if previous_process is not None and previous_process.poll() is None:
    previous_process.terminate()
    try:
        previous_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        previous_process.kill()
        previous_process.wait(timeout=5)

speed_flags = ['--fast', 'autotune']  # cuDNN tuning; no fp8 or reduced-precision accumulation.
if CHANNELS_LAST:
    speed_flags.append('--force-channels-last')
if VRAM_MODE == 'high':
    speed_flags.append('--highvram')
# In auto mode, leave ComfyUI's smart memory/dynamic VRAM defaults enabled.
print('Speed settings: FP16 diffusion/text, cuDNN autotune, VRAM mode =', VRAM_MODE)
comfy_process = subprocess.Popen([
    sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
    '--fp16-unet', '--fp16-text-enc', '--enable-cors-header',
    *attention_flag,
    *speed_flags,
], cwd=RUNTIME_ROOT, env=env)

SERVER_URL = None
try:
    deadline = time.time() + 180
    while time.time() < deadline:
        if comfy_process.poll() is not None:
            raise RuntimeError('ComfyUI exited before it opened. Rerun the previous cells and inspect the Colab output.')
        try:
            with socket.create_connection(('127.0.0.1', PORT), timeout=2):
                break
        except OSError:
            time.sleep(2)
    else:
        raise TimeoutError('ComfyUI did not open its port within three minutes.')

    if WARMUP_ON_START:
        warmup_default_checkpoint(f'http://127.0.0.1:{PORT}')

    # Optional: pin the ngrok URL so it never changes across tunnel/runtime restarts.
    # Every ngrok account gets one free dev domain automatically - find yours at
    # ngrok dashboard -> Gateway -> Domains, then add it as the NGROK_DOMAIN Colab secret
    # (looks like your-name.ngrok-free.dev). Without it you get a random URL each time
    # the tunnel reconnects, which silently strands the phone app on a dead address -
    # a failure mode that looks identical to "generation just stopped working".
    NGROK_DOMAIN = colab_secret('NGROK_DOMAIN')
    tunnel_kwargs = {'auth': f'{TUNNEL_USER}:{TUNNEL_PASSWORD}'}
    if NGROK_DOMAIN:
        tunnel_kwargs['domain'] = NGROK_DOMAIN
        print('Using reserved ngrok domain:', NGROK_DOMAIN)
    else:
        print('No NGROK_DOMAIN secret set - using a random URL that changes if the tunnel restarts.')
        print('Free fix: ngrok dashboard -> Gateway -> Domains -> copy your dev domain -> add as NGROK_DOMAIN secret.')

    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    for old_tunnel in ngrok.get_tunnels():
        ngrok.disconnect(old_tunnel.public_url)
    tunnel = ngrok.connect(PORT, 'http', **tunnel_kwargs)
    SERVER_URL = tunnel.public_url

    print('\n' + '=' * 56)
    print('Paste these into Phone Studio → Connect')
    print('Server URL:     ', SERVER_URL)
    print('Tunnel user:    ', TUNNEL_USER)
    print('Tunnel password:', TUNNEL_PASSWORD)
    print('Studio token:   ', PHONE_STUDIO_TOKEN)
    print('=' * 56)
    print('Keep this Colab tab running while you generate images.')
    print('This cell will now stay busy on purpose instead of finishing.')
    print('A finished cell can make Colab treat the runtime as idle and recycle it sooner,')
    print('so this loop keeps it occupied and reconnects the tunnel if it drops.')
    print('Stop this cell (Runtime > Interrupt) to shut everything down; rerun it to relaunch.')

    heartbeat = 0
    while True:
        time.sleep(60)
        heartbeat += 1

        if comfy_process.poll() is not None:
            raise RuntimeError('ComfyUI process exited. Rerun the previous cells, then this one, to relaunch.')

        active_urls = [t.public_url for t in ngrok.get_tunnels()]
        if SERVER_URL not in active_urls:
            print('ngrok tunnel dropped, reconnecting...')
            tunnel = ngrok.connect(PORT, 'http', **tunnel_kwargs)
            SERVER_URL = tunnel.public_url
            print('New Server URL:', SERVER_URL, '  <-- update this in Phone Studio')

        if heartbeat % 5 == 0:
            print(f'[{time.strftime("%H:%M:%S")}] still running -> {SERVER_URL}')
except KeyboardInterrupt:
    print('Stopped manually. Run this cell again to relaunch ComfyUI and get a fresh tunnel.')
finally:
    # Release GPU memory on interruption/error so a rerun cannot leave two servers.
    if comfy_process.poll() is None:
        comfy_process.terminate()
        try:
            comfy_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            comfy_process.kill()
            comfy_process.wait(timeout=5)
    try:
        if SERVER_URL:
            ngrok.disconnect(SERVER_URL)
    except Exception:
        pass
